In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os, json, subprocess
from pathlib import Path
import asyncio
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

**Bộ 3 servers hôm nay:**

- `mcp-server-fetch` — lấy nội dung từ URL bất kỳ (uvx) - chúng ta vưa làm lúc nãy
- `server-memory` — knowledge graph lưu entities và relations (node, global)
- `server-filesystem` — đọc/ghi file trên máy (node, global)

In [15]:
# Fetch
fetch_params = { # Define MCP server params
    "command": "uvx",
    "args": ["mcp-server-fetch"],
}

# Main async function
async with MCPServerStdio(
    params=fetch_params,
    client_session_timeout_seconds=30
) as server:
    fetch_tools = await server.list_tools()

    print(f"Số tools: {len(fetch_tools)}")

    for tool in fetch_tools:
        print(f"-> {tool.name}")
        print(f"-> {tool.description}")
        print()


Số tools: 1
-> fetch
-> Fetches a URL from the internet and optionally extracts its contents as markdown.

Although originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.



In [3]:
WORKSPACE = Path("agent_workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

In [4]:
# ── Tìm đường dẫn MCP Servers (NPM Global)
NPM_ROOT = subprocess.check_output(["npm", "root", "-g"]).decode().strip()
FS_PATH  = os.path.join(NPM_ROOT, "@modelcontextprotocol", "server-filesystem", "dist", "index.js")
MEM_PATH = os.path.join(NPM_ROOT, "@modelcontextprotocol", "server-memory",     "dist", "index.js")

In [ ]:
# Memory server
memory_params = {
    "command": "node",
    "args": [MEM_PATH]
}

async with MCPServerStdio(
    params=memory_params,
    client_session_timeout_seconds=30
) as server:
    memory_tools = await server.list_tools()

    print(f"Số tools: {len(memory_tools)}")

    for tool in memory_tools:
        print(f"-> {tool.name}")
        print(f"-> {tool.description}")
        print()
        

Số tools: 9
-> create_entities
-> Create multiple new entities in the knowledge graph

-> create_relations
-> Create multiple new relations between entities in the knowledge graph. Relations should be in active voice

-> add_observations
-> Add new observations to existing entities in the knowledge graph

-> delete_entities
-> Delete multiple entities and their associated relations from the knowledge graph

-> delete_observations
-> Delete specific observations from entities in the knowledge graph

-> delete_relations
-> Delete multiple relations from the knowledge graph

-> read_graph
-> Read the entire knowledge graph

-> search_nodes
-> Search for nodes in the knowledge graph based on a query

-> open_nodes
-> Open specific nodes in the knowledge graph by their names



In [10]:
# Filesystem server
fs_params = {
    "command": "node",
    "args": [FS_PATH, str(WORKSPACE)]
}

async with MCPServerStdio(
    params=fs_params,
    client_session_timeout_seconds=30
) as server:
    fs_tools = await server.list_tools()

    print(f"Số tools: {len(fs_tools)}")

    for tool in fs_tools:
        print(f"-> {tool.name}")
        print(f"-> {tool.description}")
        print()

Số tools: 14
-> read_file
-> Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

-> read_text_file
-> Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.

-> read_media_file
-> Read an image or audio file. Returns the base64 encoded data and MIME type. Only works within allowed directories.

-> read_multiple_files
-> Read the contents of multiple files simultaneously. This is more efficient than reading files one by one when you need to analyze or compare multiple files. Each file's content is returned with its path as a reference. Failed reads for indi

In [11]:
# json schema
create_tool = next(t for t in fs_tools if t.name == "read_file")
print(f"Tên: {create_tool.name}")
print(f"Mô tả: {create_tool.description}")
print(json.dumps(create_tool.inputSchema, indent=2, ensure_ascii=False))

Tên: read_file
Mô tả: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "properties": {
    "path": {
      "type": "string"
    },
    "tail": {
      "description": "If provided, returns only the last N lines of the file",
      "type": "number"
    },
    "head": {
      "description": "If provided, returns only the first N lines of the file",
      "type": "number"
    }
  },
  "required": [
    "path"
  ]
}


# Agent Nghiên cứu Việc Làm IT

In [12]:
INSTRUCTIONS = """
Bạn là chuyên gia phân tích dữ liệu IT Việt Nam.

Phong cách làm việc:
- Đọc kỹ, tổng hợp chính xác, không bịa đặt thông tin
- Chỉ lưu vào knowledge graph những gì có bằng chứng từ nội dung đã đọc
- Báo cáo cuối bằng tiếng Việt, rõ ràng, thực dụng

Ngôn ngữ: Tiếng Việt cho tất cả output hướng đến người dùng.
Tool calls: Theo đúng JSON schema (tiếng Anh).
"""

TASK = """
Thực hiện nghiên cứu thị trường việc làm IT Việt Nam qua 4 bước:

BƯỚC 1 — Fetch nội dung từ: https://itviec.com/blog/luong-it/ (Dự phòng: https://topdev.vn/blog/)

BƯỚC 2 — Xây knowledge graph kỹ năng
Tạo entities loại "ky_nang" cho từng kỹ năng/công nghệ tìm thấy trong nội dung.
Mỗi entity cần ít nhất 3 observations:
  - Mức lương trung bình (nếu có số liệu cụ thể)
  - Mức độ nhu cầu: cao / trung_binh / thap
  - Loại: backend / frontend / devops / data / mobile / ai / other

Sau đó tạo entities loại "vi_tri" cho các vị trí công việc.
Tạo relations: [ky_nang] → "required_for" → [vi_tri]

BƯỚC 3 — Lưu knowledge graph ra file
Dùng read_graph để lấy toàn bộ graph.
Ghi vào file: sandbox/kg_vieclamIT.json

BƯỚC 4 — Xuất báo cáo markdown
Đọc lại graph, rồi ghi file: sandbox/bao_cao_vieclamIT.md

Nội dung báo cáo:
# Báo Cáo Thị Trường Việc Làm IT Việt Nam
**Ngày:** [hôm nay] | **Nguồn:** [URL đã fetch]

## Tóm Tắt
[3 câu insight quan trọng nhất]

## Kỹ Năng Đang Được Tuyển Nhiều Nhất
| Kỹ năng | Mức nhu cầu | Lương TB | Loại |
|---------|------------|---------|------|

## Vị Trí Nổi Bật
[Mô tả 3-4 vị trí: yêu cầu kỹ năng gì, lương range]

## Khuyến Nghị
[2-3 gợi ý cụ thể để tăng cơ hội việc làm]

Sau khi hoàn tất, xác nhận tổng số entities và relations đã tạo.
"""

In [16]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=30) as mcp_fetch:
    async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=15) as mcp_memory:
        async with MCPServerStdio(params=fs_params, client_session_timeout_seconds=15) as mcp_files:

            agent = Agent(
                name="MarketAnalyzer",
                instructions=INSTRUCTIONS,
                model="gpt-4o-mini",
                mcp_servers=[mcp_fetch, mcp_memory, mcp_files]
            )

            with trace("it-market-research"):
                result = await Runner.run(agent, TASK, max_turns=25)
                print(result.final_output)
    

Đã hoàn thành nghiên cứu thị trường việc làm IT Việt Nam. Dưới đây là thông tin chi tiết:

### Tổng Kết
- **Ngày**: 2023-10-10
- **Nguồn**: [ITViec](https://itviec.com/blog/luong-it/)

#### Tóm Tắt
1. Ngành Dược phẩm có mức lương trung bình cao nhất cho chuyên gia IT tại Việt Nam.
2. Nhu cầu về kỹ sư Backend và AI đang gia tăng trong thị trường.
3. Các công ty Consulting trả lương cao hơn so với các loại hình công ty khác.

#### Kỹ Năng Đang Được Tuyển Nhiều Nhất
| Kỹ năng              | Mức nhu cầu | Lương TB         | Loại     |
|---------------------|-------------|------------------|----------|
| Backend             | Cao         | 50 triệu đồng/tháng | backend  |
| Frontend            | Cao         | 45 triệu đồng/tháng | frontend |
| DevOps              | Trung bình  | 55 triệu đồng/tháng | devops   |
| Data Science        | Cao         | 60 triệu đồng/tháng | data     |
| Mobile Development  | Trung bình  | 40 triệu đồng/tháng | mobile   |
| AI                  | Cao         | 70